<a href="https://colab.research.google.com/github/zfifteen/unified-framework/blob/main/notebooks/z5d_prime_plus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import math
import numpy as np
PHI = (1 + math.sqrt(5)) / 2

def bootstrap_enhancement(N, k, bins=50, resamples=1000):
    numbers = np.arange(1, N+1)
    is_prime = np.ones(N+1, dtype=bool)
    is_prime[0] = is_prime[1] = False
    for i in range(2, int(np.sqrt(N)) + 1):
        if is_prime[i]:
            is_prime[i*i::i] = False
    is_prime_flags = is_prime[numbers]
    frac = np.fmod(numbers, PHI) / PHI
    angles = PHI * np.power(frac, k)  # Removed % (2*pi) for direct range [0, PHI]
    enhancements = []
    for _ in range(resamples):
        sample_indices = np.random.randint(0, N, size=N)
        sample_angles = angles[sample_indices]
        sample_primes = is_prime_flags[sample_indices]
        prime_angles = sample_angles[sample_primes]
        hist_all, bin_edges = np.histogram(sample_angles, bins=bins)
        hist_all = np.maximum(hist_all, 1)
        hist_prime, _ = np.histogram(prime_angles, bin_edges)
        density_prime = hist_prime / hist_all.astype(float)
        average_density = np.sum(sample_primes) / N
        if average_density == 0:
            continue
        max_density = np.max(density_prime)
        enhancement = (max_density / average_density - 1) * 100
        enhancements.append(enhancement)
    enh = np.array(enhancements)
    mean = np.mean(enh)
    ci_low = np.percentile(enh, 2.5)
    ci_high = np.percentile(enh, 97.5)
    return mean, ci_low, ci_high, enhancements

# Example usage
N = 1000000
k_base = 0.3
k_mod = 1.618
mean_base, ci_low_base, ci_high_base, enhancements_base = bootstrap_enhancement(N, k_base)
mean_mod, ci_low_mod, ci_high_mod, enhancements_mod = bootstrap_enhancement(N, k_mod)

print(f"Bootstrap Enhancement Statistics for k = {k_base}:")
print(f"  Mean Enhancement: {mean_base:.4f}%")
print(f"  95% Confidence Interval: ({ci_low_base:.4f}%, {ci_high_base:.4f}%)")
print(f"  Individual Enhancement Samples (first 10): {enhancements_base[:10]}")

print(f"\nBootstrap Enhancement Statistics for k = {k_mod}:")
print(f"  Mean Enhancement: {mean_mod:.4f}%")
print(f"  95% Confidence Interval: ({ci_low_mod:.4f}%, {ci_high_mod:.4f}%)")
print(f"  Individual Enhancement Samples (first 10): {enhancements_mod[:10]}")

Bootstrap Enhancement Statistics for k = 0.3:
  Mean Enhancement: 89.5112%
  95% Confidence Interval: (15.4946%, 246.6884%)
  Individual Enhancement Samples (first 10): [np.float64(130.76301791874835), np.float64(121.32139496219389), np.float64(39.16775594428976), np.float64(29.05147900369307), np.float64(169.48748196960315), np.float64(32.98967260024184), np.float64(59.701653651474814), np.float64(188.55998180917877), np.float64(50.90695077415266), np.float64(38.93037504254744)]

Bootstrap Enhancement Statistics for k = 1.618:
  Mean Enhancement: 7.6450%
  95% Confidence Interval: (5.1567%, 11.6243%)
  Individual Enhancement Samples (first 10): [np.float64(7.841410122065162), np.float64(5.566580960783507), np.float64(12.454494800993476), np.float64(7.8187913159713185), np.float64(5.288843528916498), np.float64(9.01553590185582), np.float64(6.927727512179027), np.float64(8.00433190633736), np.float64(6.010066313444384), np.float64(5.72433525193019)]


In [22]:
import math
import numpy as np

import math

def z5d_prime(n):
    if n < 2:
        return 0
    x = float(n)
    logx = math.log(x)
    loglogx = math.log(logx)
    # 5-term asymptotic for p_n (Riemann-inspired, calibrated for Z_5D)
    term1 = logx
    term2 = loglogx - 1
    term3 = (loglogx - 2) / logx
    term4 = (loglogx**2 - 6 * loglogx + 11) / (2 * logx**2)
    term5 = (loglogx**3 - 12.5 * loglogx**2 + 47 * loglogx - 59.5) / (6 * logx**3)
    series = term1 + term2 + term3 + term4 + term5
    return int(x * series)

# Example benchmark
n_values = [10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9, 10**10]
true_p = [7919, 104729, 1299709, 15485863, 179424673, 2032802211, 22801763489, 255275268861]
for i, n in enumerate(n_values):
    pred = z5d_prime(n)
    true = true_p[i]
    error = abs(pred - true) / true * 100
    band_label = f"10^{int(math.log10(n))}"
    disclaimer = ""
    if n == 10**10:
        disclaimer = " (extrapolated)"
    print(f"{band_label} (n={n}): Predicted={pred}, True={true}, Error={error:.4f}%{disclaimer}")

10^3 (n=1000): Predicted=7859, True=7919, Error=0.7577%
10^4 (n=10000): Predicted=104687, True=104729, Error=0.0401%
10^5 (n=100000): Predicted=1300311, True=1299709, Error=0.0463%
10^6 (n=1000000): Predicted=15491941, True=15485863, Error=0.0392%
10^7 (n=10000000): Predicted=179502122, True=179424673, Error=0.0432%
10^8 (n=100000000): Predicted=2038659734, True=2032802211, Error=0.2882%
10^9 (n=1000000000): Predicted=22806540226, True=22801763489, Error=0.0209%
10^10 (n=10000000000): Predicted=252136775283, True=255275268861, Error=1.2295% (extrapolated)


In [23]:
import mpmath as mp
mp.mp.dps = 50  # For Δ_n < 10^{-16}

def li(x):
    return mp.li(x)  # Logarithmic integral

def z5d_prime_plus(n, zeta_zeros):
    if n < 2:
        return mp.mpf(0)
    x_approx = n * mp.log(n)  # Base PNT inverse start
    print(f"  Approximate x (n * log(n)): {x_approx}")
    # 5-term asymptotic base (from prior impl)
    logx = mp.log(x_approx)
    loglogx = mp.log(logx)
    series = logx + loglogx - 1 + (loglogx - 2)/logx + (loglogx**2 - 6*loglogx + 11)/(2*logx**2)
    base_pred = x_approx * (1 + 1/logx + 2/logx**2 + 6/logx**3 + 24/logx**4)  # Refined
    print(f"  Base Prediction (5-term PNT): {base_pred}")
    # Explicit correction with zeta zeros
    correction = mp.mpf(0)
    log_x = mp.log(x_approx)
    print(f"  Calculating correction using first {len(zeta_zeros[:100000])} zeta zeros...")
    for i, t in enumerate(zeta_zeros[:100000]):  # Use full 100k for production
        rho = mp.mpc(0.5, t)
        correction_term = mp.re(mp.ei(rho * log_x))  # Approx li(x^rho) ~ ei(rho log x)
        correction += correction_term
        if (i + 1) % 10000 == 0:
             print(f"    Processed {i + 1} zeros, current correction sum: {correction}")

    enhanced_pred = base_pred - correction  # Simplified; full formula includes 1/rho factor and pairs
    print(f"  Total Zeta Correction (simplified): {correction}")
    print(f"  Enhanced Prediction (Base - Correction): {enhanced_pred}")
    final_pred = mp.nint(enhanced_pred)
    print(f"  Final Rounded Prediction: {final_pred}")
    return final_pred

In [24]:
import mpmath as mp
mp.mp.dps = 50

def approx_zeta_correction(x, num_terms=1000):
    # Asymptotic approx: sum over zeros ~ integral + fluctuation; here use average density
    logx = mp.log(x)
    sum_approx = mp.mpf(0)
    avg_gap = mp.mpf(3.0)  # From stats; scales with log T
    for k in range(1, num_terms + 1):
        t_k = avg_gap * mp.log(k + 1)  # Better than linear: log-scaled for density
        rho = mp.mpc(0.5, t_k)
        sum_approx += mp.re(mp.ei(rho * logx)) / mp.norm(rho)  # Approx li(x^rho) ~ ei(rho log x)
    return -sum_approx  # Simplified; add conjugates and 1/rho factor for full

# Integrate into Z_5D
def z5d_prime_approx(n):
    x_approx = mp.mpf(n) * mp.log(n)
    base = x_approx * (1 + 1/mp.log(x_approx) + 2/mp.log(x_approx)**2)  # 3-term for demo
    correction = approx_zeta_correction(x_approx)
    enhanced_pred = base + correction
    final_pred = mp.nint(enhanced_pred)
    return {
        "n": n,
        "x_approx": x_approx,
        "base_prediction": base,
        "zeta_correction": correction,
        "enhanced_prediction": enhanced_pred,
        "final_prediction": final_pred
    }

# Example usage and printing stats
n_value_for_stats = 10**8  # Choose a value of n to demonstrate
stats = z5d_prime_approx(n_value_for_stats)

print(f"Statistics for n = {stats['n']}:")
print(f"  Approximate x (n * log(n)): {stats['x_approx']}")
print(f"  Base Prediction (3-term PNT): {stats['base_prediction']}")
print(f"  Approximate Zeta Correction: {stats['zeta_correction']}")
print(f"  Enhanced Prediction (Base + Correction): {stats['enhanced_prediction']}")
print(f"  Final Rounded Prediction: {stats['final_prediction']}")

Statistics for n = 100000000:
  Approximate x (n * log(n)): 1842068074.395236547214393163747491366080881190903
  Base Prediction (3-term PNT): 1936506093.2812282618775619241896148189107392183129
  Approximate Zeta Correction: -768.71422502767127747759560149784497513282431010409
  Enhanced Prediction (Base + Correction): 1936505324.5670032342062844465940133210657640854886
  Final Rounded Prediction: 1936505325.0


Self-contained Gram series for pi(x) (invert for p_n via binary search; mpmath dps=50):

In [25]:
import mpmath as mp
mp.mp.dps = 50

def gram_pi(x, terms=20):
    logx = mp.log(x)
    sum_gram = mp.mpf(1)  # k=0 term
    for k in range(1, terms + 1):
        sum_gram += (logx ** k) / (mp.factorial(k) * k * mp.zeta(k + 1))
    return sum_gram

def z5d_prime_gram(n, terms=20):
    if n < 2: return mp.mpf(0)
    # Binary search for x where gram_pi(x) ≈ n
    low = mp.mpf(n)
    high = mp.mpf(n) * mp.log(n) * mp.mpf(1.2)  # Upper bound

    stats = []
    for i in range(50):  # Converges fast
        mid = (low + high) / 2
        gram_mid = gram_pi(mid, terms)
        stats.append({
            "iteration": i,
            "low": low,
            "high": high,
            "mid": mid,
            "gram_pi(mid)": gram_mid,
            "target_n": n
        })
        if gram_mid < n:
            low = mid
        else:
            high = mid
    final_pred = mp.nint(low)
    return final_pred, stats

# Example usage and printing stats
n_value_for_stats = 10**8  # Choose a value of n to demonstrate
final_prediction, stats_data = z5d_prime_gram(n_value_for_stats)

print("Gram series for pi(x) (invert for p_n via binary search; mpmath dps=50):")
print(f"Binary Search Statistics for n = {n_value_for_stats}:")
for entry in stats_data:
    print(f"  Iteration {entry['iteration']}: low={entry['low']}, high={entry['high']}, mid={entry['mid']}, gram_pi(mid)={entry['gram_pi(mid)'].__str__()}, target={entry['target_n']}")

print(f"\nFinal Rounded Prediction for n = {n_value_for_stats}: {final_prediction}")

Gram series for pi(x) (invert for p_n via binary search; mpmath dps=50):
Binary Search Statistics for n = 100000000:
  Iteration 0: low=100000000.0, high=2210481689.2742837748530162316762938289998815086171, mid=1155240844.6371418874265081158381469144999407543086, gram_pi(mid)=33457229.761068430088097347108442381802536079007928, target=100000000
  Iteration 1: low=1155240844.6371418874265081158381469144999407543086, high=2210481689.2742837748530162316762938289998815086171, mid=1682861266.9557128311397621737572203717499111314628, gram_pi(mid)=45108092.636767291585160679111155683141679596845894, target=100000000
  Iteration 2: low=1682861266.9557128311397621737572203717499111314628, high=2210481689.2742837748530162316762938289998815086171, mid=1946671478.11499830299638920271675710037489632004, gram_pi(mid)=50597565.680302031567239053509710166078428593789874, target=100000000
  Iteration 3: low=1946671478.11499830299638920271675710037489632004, high=2210481689.27428377485301623167629382899

GUE-inspired zero simulation (for Monte-Carlo correction; avg_gap ~ ln(T)/(2π)):

In [26]:
import numpy as np
import mpmath as mp
mp.mp.dps = 50

def monte_carlo_correction(x, num_samples=1000):
    logx = mp.log(x)
    T = mp.mpf(1000)  # Upper Im rho; scale with sqrt(x)
    # Simulate GUE spacings: eigenvalues of random Hermitian
    gaps = np.abs(np.random.normal(0, 1, num_samples))  # Simplified Poisson approx
    gaps = np.sort(gaps * (mp.log(T) / (2 * mp.pi)))  # Scale
    corrections = []
    for t in gaps:
        rho = mp.mpc(0.5, mp.mpf(t))
        correction_term = mp.re(mp.li(x ** rho)) / rho  # Pair conjugates implicitly
        corrections.append(correction_term)
    total_correction = -sum(corrections) / num_samples  # Average
    return total_correction, corrections  # Return total correction and individual terms

# Integrate: base + monte_carlo_correction(x_approx)

# Example usage and printing stats
n_value_for_stats = 10**8  # Choose a value of n to demonstrate
x_approx_for_stats = mp.mpf(n_value_for_stats) * mp.log(n_value_for_stats)
total_correction, individual_corrections = monte_carlo_correction(x_approx_for_stats)

print("GUE-inspired zero simulation (for Monte-Carlo correction; avg_gap ~ ln(T)/(2π)):")
print(f"Monte Carlo Correction Statistics for x_approx = {x_approx_for_stats}:")
print(f"  Total Approximate Zeta Correction (averaged over {len(individual_corrections)} samples): {total_correction}")
print("\nIndividual Correction Terms (first 10):")
for i, term in enumerate(individual_corrections[:10]):
    print(f"  Term {i+1}: {term}")

GUE-inspired zero simulation (for Monte-Carlo correction; avg_gap ~ ln(T)/(2π)):
Monte Carlo Correction Statistics for x_approx = 1842068074.395236547214393163747491366080881190903:
  Total Approximate Zeta Correction (averaged over 1000 samples): (-404.79311382782687627357129247202467152969712745661 + 434.24963972920766855642029397261774853406076320342j)

Individual Correction Terms (first 10):
  Term 1: (9011.5568229252715578881747948876620585709103862007 - 26.302828830503661362398171667532290468733296352677j)
  Term 2: (9011.4841340095333936198300985960419409200796735737 - 26.566034575267336342586504689764802274113216174097j)
  Term 3: (8925.5749177321951341980340137057679446515274361676 - 129.87896172976105559644042192185975579950713953924j)
  Term 4: (8908.3330444782401285840403632366768926325914969072 - 141.57839347794572225247457431700954036995030703136j)
  Term 5: (8789.962810937799560761224931290995412407381975272 - 203.07692749339214255539823660247827533518163329769j)
  Term 